In [2]:
import requests
import json 

In [27]:
GROQ_API_KEY = "####"

In [17]:
def call_llm(prompt: str , temperature: float = 0.1)-> dict:
    response = requests.post(
       "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {GROQ_API_KEY}"
        },
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 500,
            "temperature": temperature
        }
    )

    data = response.json()
    return {
        "response": data["choices"][0]["message"]["content"],
        "tokens_used": data["usage"]["completion_tokens"]
    }


In [5]:
# Financial context
context = """Zomato Q3 FY24 Results:
Revenue: Rs 3,288 crore
Operating Expenses: Rs 3,102 crore  
Interest Expense: Rs 48 crore
Tax: Rs 0 crore"""

In [6]:
# Prompt A — standard prompt, no CoT

In [11]:
prompt_standard = f"""You are a financial analyst. Using only the context below, answer the question.

Context: {context}

Question: Is Zomato profitable? By how much?
Answer:"""

In [15]:
# Prompt B — CoT prompt
prompt_cot = f"""You are a financial analyst. Using only the context below, answer the question.
Think through this step by step before giving your final answer.

Context: {context}

Question: Is Zomato profitable? By how much?

Let's think step by step:"""

In [18]:
print("Running standard prompt...")
result_standard = call_llm(prompt_standard)
print(f"Response: {result_standard['response']}")
print(f"Tokens used: {result_standard['tokens_used']}")
print()
print("Running CoT prompt...")
result_cot = call_llm(prompt_cot)
print(f"Response: {result_cot['response']}")
print(f"Tokens used: {result_cot['tokens_used']}")

Running standard prompt...
Response: To determine if Zomato is profitable, we need to calculate its net income. 

Revenue: Rs 3,288 crore
Operating Expenses: Rs 3,102 crore
Interest Expense: Rs 48 crore
Tax: Rs 0 crore

First, let's calculate the net income from operations:
Revenue - Operating Expenses = Rs 3,288 crore - Rs 3,102 crore = Rs 186 crore

Then, let's subtract the interest expense:
Net income from operations - Interest Expense = Rs 186 crore - Rs 48 crore = Rs 138 crore

Since tax is Rs 0 crore, the net income is the same as the net income from operations minus interest expense.

Therefore, Zomato is profitable by Rs 138 crore.
Tokens used: 157

Running CoT prompt...
Response: To determine if Zomato is profitable, we need to calculate its net income. 

Step 1: Calculate the total revenue. 
Total revenue = Rs 3,288 crore

Step 2: Calculate the total operating expenses. 
Total operating expenses = Rs 3,102 crore

Step 3: Calculate the operating profit. 
Operating profit = Tot

In [19]:
# Complex multi-step reasoning — where CoT actually matters
complex_context = """Zomato Q2 FY24 Results:
Revenue: Rs 2,860 crore
Total Costs: Rs 2,870 crore"""

complex_question = "Zomato's revenue grew 15% QoQ and costs grew 8% QoQ. Will they be profitable in Q3?"

prompt_complex_standard = f"""You are a financial analyst. Using only the context below, answer the question.

Context: {complex_context}

Question: {complex_question}
Answer:"""

In [20]:
prompt_complex_cot = f"""You are a financial analyst. Using only the context below, answer the question.
Think through this step by step before giving your final answer.

Context: {complex_context}

Question: {complex_question}

Let's think step by step:"""

In [21]:
print("Standard prompt on complex question...")
result_s = call_llm(prompt_complex_standard)
print(f"Response: {result_s['response']}")
print(f"Tokens used: {result_s['tokens_used']}")
print()
print("CoT prompt on complex question...")
result_c = call_llm(prompt_complex_cot)
print(f"Response: {result_c['response']}")
print(f"Tokens used: {result_c['tokens_used']}")

Standard prompt on complex question...
Response: Based on the given information, Zomato's revenue grew 15% QoQ and costs grew 8% QoQ. However, their total costs (Rs 2,870 crore) are higher than their revenue (Rs 2,860 crore). 

Given that costs are higher than revenue, it is unlikely that Zomato will be profitable in Q3.
Tokens used: 78

CoT prompt on complex question...
Response: To determine if Zomato will be profitable in Q3, let's break down the information step by step:

1. Revenue growth: Zomato's revenue grew 15% QoQ. This means their revenue for Q2 is Rs 2,860 crore, but we don't know the revenue for Q1. However, we can infer that their revenue for Q1 was lower than Rs 2,860 crore.

2. Costs growth: Zomato's costs grew 8% QoQ. This means their costs for Q2 are Rs 2,870 crore, but we don't know the costs for Q1. However, we can infer that their costs for Q1 were lower than Rs 2,870 crore.

3. Current situation: Zomato's revenue for Q2 is Rs 2,860 crore, and their costs for Q2 ar

In [24]:
# Structured CoT — tell the model exactly what steps to take
prompt_structured_cot = f"""You are a financial analyst. Using only the context below, answer the question.
Follow these exact calculation steps:

Step 1: Calculate Q3 Revenue = Q2 Revenue × (1 + revenue growth rate)
Step 2: Calculate Q3 Costs = Q2 Costs × (1 + cost growth rate)  
Step 3: Calculate Q3 Profit = Q3 Revenue - Q3 Costs
Step 4: State whether profitable and by how much

Context: {complex_context}

Question: {complex_question}

Let me work through each step:"""

In [25]:
print("Structured CoT prompt...")
result_structured = call_llm(prompt_structured_cot)
print(f"Response: {result_structured['response']}")
print(f"Tokens used: {result_structured['tokens_used']}")
print(f"\nCorrect answer: Rs 189.4 crore profit")

Structured CoT prompt...
Response: To determine whether Zomato will be profitable in Q3, we'll follow the steps provided.

Step 1: Calculate Q3 Revenue = Q2 Revenue × (1 + revenue growth rate)
Given Q2 Revenue = Rs 2,860 crore and revenue growth rate = 15% (or 0.15 as a decimal)
Q3 Revenue = Rs 2,860 crore × (1 + 0.15)
Q3 Revenue = Rs 2,860 crore × 1.15
Q3 Revenue = Rs 3,293 crore

Step 2: Calculate Q3 Costs = Q2 Costs × (1 + cost growth rate)
Given Q2 Costs = Rs 2,870 crore and cost growth rate = 8% (or 0.08 as a decimal)
Q3 Costs = Rs 2,870 crore × (1 + 0.08)
Q3 Costs = Rs 2,870 crore × 1.08
Q3 Costs = Rs 3,089.6 crore

Step 3: Calculate Q3 Profit = Q3 Revenue - Q3 Costs
Q3 Profit = Rs 3,293 crore - Rs 3,089.6 crore
Q3 Profit = Rs 203.4 crore

Step 4: State whether profitable and by how much
Zomato will be profitable in Q3 by Rs 203.4 crore.
Tokens used: 290

Correct answer: Rs 189.4 crore profit


In [26]:
def should_use_cot(query: str) -> dict:
    """
    WHY: CoT costs more tokens. Only use it when it actually helps.
    Decision based on query complexity signals.
    """
    
    cot_triggers = [
        "calculate", "compare", "grew", "increased", "decreased",
        "by how much", "difference", "trend", "percent", "%",
        "will", "forecast", "predict"
    ]
    
    query_lower = query.lower()
    triggered = [t for t in cot_triggers if t in query_lower]
    use_cot = len(triggered) > 0
    
    return {
        "use_cot": use_cot,
        "triggers_found": triggered,
        "prompt_type": "structured_cot" if use_cot else "standard"
    }

# Test on our three query types
test_queries = [
    "Is Zomato profitable?",
    "Zomato revenue grew 15% QoQ, will they be profitable in Q3?",
    "What is Zomato's PAT for Q3 FY24?",
    "Should I buy Zomato stock?"
]

for q in test_queries:
    result = should_use_cot(q)
    print(f"Query: {q}")
    print(f"  Use CoT: {result['use_cot']} | Triggers: {result['triggers_found']}")
    print()

Query: Is Zomato profitable?
  Use CoT: False | Triggers: []

Query: Zomato revenue grew 15% QoQ, will they be profitable in Q3?
  Use CoT: True | Triggers: ['grew', '%', 'will']

Query: What is Zomato's PAT for Q3 FY24?
  Use CoT: False | Triggers: []

Query: Should I buy Zomato stock?
  Use CoT: False | Triggers: []



## Day 59 — Prompt Failure Modes + Chain of Thought

### What I built
- Standard vs CoT comparison on simple and complex financial questions
- Discovered generic CoT fails for mathematical reasoning
- Built structured CoT with explicit calculation steps
- Built keyword-based CoT router (and found its limits)

### Key findings
1. Simple problems: CoT wastes tokens, standard prompt sufficient
2. Complex problems + generic CoT: reasoning without calculation = wrong answer
3. Complex problems + structured CoT: correct reasoning, but LLM arithmetic unreliable
4. Never let LLMs do critical financial math — extract numbers, compute in Python

### CoT decision rule
- Simple factual queries → standard prompt
- Multi-step reasoning / calculations → structured CoT
- Production solution: LLM classifier routes queries, not keywords

### Critical production rule
LLMs handle reasoning and language.
Python handles arithmetic.
Never mix these responsibilities.

### Token cost reality
Standard prompt: ~78-157 tokens
Generic CoT: ~338 tokens  
Structured CoT: ~290 tokens
CoT costs 2-4x more — only use when it changes the answer